# EDA dữ liệu thương mại điện tử

In [ ]:
%matplotlib inline
from math import ceil
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent.parent

data_dir = repo_root / "data" / "generator" / "eda" / "raw"

## 1. Olist — đơn hàng

In [ ]:
orders = pl.scan_csv(
    data_dir / "olist" / "olist_orders_dataset.csv", infer_schema=False,
).select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_delivered_customer_date", "order_estimated_delivery_date",
).with_columns(
    pl.col("order_purchase_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("order_delivered_customer_date").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("order_estimated_delivery_date").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
).collect(engine="streaming")

In [ ]:
orders.head()

In [ ]:
orders.select(
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date",
).describe()

In [ ]:
orders.null_count()

In [ ]:
orders.select(pl.col("order_id").is_duplicated().sum().alias("duplicate_order_id"))

### 1.1. Giỏ hàng và giá trị đơn

Năm 2017. Tiền hàng, phí vận chuyển và giá trị thanh toán ghi nhận được tính riêng; bảng con được gộp về mỗi đơn trước khi join.

In [ ]:
orders_2017 = orders.filter(pl.col("order_purchase_timestamp").dt.year() == 2017)

items = pl.scan_csv(
    data_dir / "olist" / "olist_order_items_dataset.csv", infer_schema=False,
).select("order_id", "product_id", "price", "freight_value").with_columns(
    pl.col("price", "freight_value").cast(pl.Float64, strict=False),
).collect(engine="streaming")

payments = pl.scan_csv(
    data_dir / "olist" / "olist_order_payments_dataset.csv", infer_schema=False,
).select("order_id", "payment_type", "payment_value").with_columns(
    pl.col("payment_value").cast(pl.Float64, strict=False),
).collect(engine="streaming")

reviews = pl.scan_csv(
    data_dir / "olist" / "olist_order_reviews_dataset.csv", infer_schema=False,
).select("order_id", "review_score").with_columns(
    pl.col("review_score").cast(pl.Float64, strict=False),
).collect(engine="streaming")

In [ ]:
order_items = items.group_by("order_id").agg(
    item_lines=pl.len(),
    products=pl.col("product_id").n_unique(),
    goods_value=pl.col("price").sum(),
    freight_value=pl.col("freight_value").sum(),
)
order_payments = payments.group_by("order_id").agg(
    payment_total=pl.col("payment_value").sum(),
)
order_reviews = reviews.group_by("order_id").agg(
    review_score=pl.col("review_score").mean(),
)

order_metrics = orders_2017.join(
    order_items, on="order_id", how="left", validate="1:1",
).join(
    order_payments, on="order_id", how="left", validate="1:1",
).join(
    order_reviews, on="order_id", how="left", validate="1:1",
)

order_metrics.select(
    "item_lines", "products", "goods_value", "freight_value", "payment_total",
).describe()

In [ ]:
basket_distribution = order_metrics.filter(pl.col("products").is_not_null()).group_by(
    "products",
).len(name="orders").sort("products").with_columns(
    share_pct=100.0 * pl.col("orders") / pl.col("orders").sum(),
)

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(
    x=basket_distribution["products"].to_list(), y=basket_distribution["share_pct"].to_numpy(),
    color="#4C72B0", ax=ax,
)
ax.set(xlabel="Số sản phẩm khác nhau/đơn", ylabel="% đơn có dòng hàng", title="Giỏ hàng — Olist 2017")
fig.tight_layout()
plt.show()

**Nhận xét:** 96,67% đơn có dòng hàng chỉ chứa một sản phẩm khác nhau; tiền hàng trung bình 138,09 cao hơn trung vị 84,99, cho thấy giá trị đơn lệch về phía cao.

**Quy luật đề xuất:** Ưu tiên giỏ nhỏ, giữ một phần đơn giá trị cao; đo riêng số sản phẩm và số lượng mỗi sản phẩm. Giá nguồn không dùng trực tiếp cho VND.

### 1.2. Trạng thái và thanh toán

Trạng thái là snapshot ghi nhận. Một đơn có thể dùng nhiều phương thức; tỷ trọng thanh toán dưới đây tính theo giá trị ghi nhận, không ánh xạ sang COD/VietQR.

In [ ]:
status_counts = orders_2017.group_by("order_status").len(name="orders").sort(
    "orders", descending=True,
).with_columns(share_pct=100.0 * pl.col("orders") / pl.col("orders").sum())

payment_mix = payments.join(
    orders_2017.select("order_id"), on="order_id", how="inner", validate="m:1",
).group_by("payment_type").agg(
    orders=pl.col("order_id").n_unique(),
    amount=pl.col("payment_value").sum(),
).sort("amount", descending=True).with_columns(
    value_pct=100.0 * pl.col("amount") / pl.col("amount").sum(),
)

payment_mix.select(
    "payment_type", "orders", pl.col("amount", "value_pct").round(2),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(
    x=status_counts["share_pct"].to_numpy(), y=status_counts["order_status"].to_list(),
    color="#4C72B0", ax=axes[0],
)
sns.barplot(
    x=payment_mix["value_pct"].to_numpy(), y=payment_mix["payment_type"].to_list(),
    color="#55A868", ax=axes[1],
)
axes[0].set(title="Trạng thái đơn — 2017", xlabel="% đơn", ylabel="", xlim=(0, 105))
axes[1].set(title="Giá trị thanh toán — 2017", xlabel="% giá trị ghi nhận", ylabel="", xlim=(0, 105))
for ax in axes:
    ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
fig.tight_layout()
plt.show()

**Nhận xét:** Thẻ tín dụng chiếm 77,76% giá trị thanh toán ghi nhận, boleto chiếm 19,26%. Đây là tỷ trọng giá trị, không phải xác suất chọn phương thức của từng đơn.

**Quy luật đề xuất:** Mô hình hóa phương thức thanh toán theo thị trường; cần bằng chứng VN riêng để chọn tỷ lệ COD/VietQR. Snapshot trạng thái chưa cho biết quá trình chuyển trạng thái.

### 1.3. Giao hàng mất bao lâu, bao nhiêu đơn trễ?

Chỉ đơn `delivered` có đủ ngày và thời gian giao không âm. Trễ là ngày giao sau ngày hẹn; giao trong ngày hẹn vẫn đúng hạn.

In [ ]:
delivered_orders = order_metrics.filter(
    (pl.col("order_status") == "delivered")
    & (pl.col("order_delivered_customer_date") >= pl.col("order_purchase_timestamp"))
    & pl.col("order_estimated_delivery_date").is_not_null(),
).with_columns(
    delivery_days=(
        pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")
    ).dt.total_seconds() / 86400,
    delay_days=(
        pl.col("order_delivered_customer_date").dt.date()
        - pl.col("order_estimated_delivery_date").dt.date()
    ).dt.total_days(),
    late=(
        pl.col("order_delivered_customer_date").dt.date()
        > pl.col("order_estimated_delivery_date").dt.date()
    ),
)

delivered_orders.select("delivery_days", "delay_days").describe()

In [ ]:
delivered_orders.select(
    orders=pl.len(),
    late_pct=(100.0 * pl.col("late").mean()).round(2),
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=delivered_orders["delivery_days"].to_numpy(), ax=ax)
ax.set(xlabel="Thời gian giao (ngày)", ylabel="Tỷ lệ đơn tích lũy", title="Thời gian giao — Olist 2017")
fig.tight_layout()
plt.show()

**Nhận xét:** Trong 43.426 đơn giao hợp lệ, thời gian giao trung vị 11,03 ngày và 5,65% giao trễ.

**Quy luật đề xuất:** Sinh riêng thời gian giao thực tế và ngày hẹn để có cả đơn đúng hẹn/trễ; cần hiệu chỉnh phân phối theo giao nhận VN.

### 1.4. Giao trễ liên quan thế nào đến đánh giá?

Chỉ đơn đủ điều kiện giao hàng và có đánh giá; điểm là trung bình trong từng đơn. Đây là liên hệ quan sát, không kết luận nhân quả.

In [ ]:
review_comparison = delivered_orders.filter(pl.col("review_score").is_not_null()).group_by(
    "late",
).agg(
    orders=pl.len(),
    mean_score=pl.col("review_score").mean(),
    low_score_orders_pct=100.0 * (pl.col("review_score") <= 2).mean(),
).sort("late").with_columns(
    pl.col("mean_score", "low_score_orders_pct").round(2),
)

review_comparison

**Nhận xét:** Điểm trung bình của đơn đúng hẹn là 4,28, so với 2,32 ở nhóm trễ; tỷ lệ điểm trung bình/đơn ≤2 lần lượt là 9,16% và 61,49%.

**Quy luật đề xuất:** Tham khảo phân phối đánh giá có điều kiện theo giao nhận; đây là liên hệ quan sát, chưa chứng minh nhân quả hoặc xác định đủ trọng số điểm 1–5.

## 2. H&M — dòng mua sản phẩm

In [ ]:
hm_purchases = pl.scan_csv(
    data_dir / "hm" / "transactions_train.csv", infer_schema=False,
).select(
    "t_dat", "customer_id", "article_id", "price", "sales_channel_id",
).with_columns(
    pl.col("t_dat").str.to_date("%Y-%m-%d", strict=False),
    pl.col("price").cast(pl.Float64, strict=False),
)

In [ ]:
hm_purchases.head()

In [ ]:
hm_purchases.select("price", "t_dat").collect(engine="streaming").describe()

In [ ]:
hm_purchases.null_count().collect(engine="streaming")

### 2.1. Mua lại

Khảo sát năm 2019. Một lần quan sát là một ngày khách có mua; mẫu số chỉ gồm khách đã mua trong năm.

In [ ]:
hm_2019 = hm_purchases.filter(pl.col("t_dat").dt.year() == 2019)

purchase_days = hm_2019.select("customer_id", "t_dat").unique().collect(engine="streaming")
customer_frequency = purchase_days.group_by("customer_id").len(name="purchase_days")

customer_frequency.select("purchase_days").describe()

In [ ]:
customer_frequency.select(
    repeat_customer_pct=((pl.col("purchase_days") > 1).mean() * 100).round(2),
)

In [ ]:
purchase_gaps = purchase_days.sort("customer_id", "t_dat").with_columns(
    gap_days=pl.col("t_dat").diff().over("customer_id").dt.total_days(),
).filter(pl.col("gap_days").is_not_null())

purchase_gaps.select("gap_days").describe()

**Nhận xét:** Trong nhóm khách đã mua năm 2019, 66,20% mua ở ít nhất hai ngày; khoảng cách giữa các ngày mua có trung vị 19 ngày, trung bình 35,26 ngày.

**Quy luật đề xuất:** Sinh tần suất và khoảng cách mua không đồng đều giữa khách; ngày mua không tương đương số đơn, cần kiểm tra đầu ra sau khi phân bổ số đơn.

### 2.2. Sản phẩm bán tập trung đến mức nào?

Đếm dòng mua theo sản phẩm trong 2019; đối chiếu với cách chọn sản phẩm đều nhau.

In [ ]:
product_counts = hm_2019.group_by("article_id").len(name="purchases").collect(
    engine="streaming",
).sort("purchases", descending=True).with_row_index("rank", offset=1)

product_counts = product_counts.with_columns(
    products_pct=100.0 * pl.col("rank") / product_counts.height,
    purchases_pct=100.0 * pl.col("purchases").cum_sum() / pl.col("purchases").sum(),
)

product_counts.filter(
    pl.col("rank").is_in([ceil(product_counts.height * share) for share in (0.01, 0.05, 0.1, 0.2)]),
).select("rank", "products_pct", "purchases_pct").with_columns(
    pl.col("products_pct", "purchases_pct").round(2),
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(product_counts["products_pct"], product_counts["purchases_pct"], label="H&M 2019")
ax.plot([0, 100], [0, 100], "--", color="gray", label="Phân bố đều")
ax.set(xlabel="% sản phẩm, xếp từ bán nhiều đến ít", ylabel="% dòng mua tích lũy")
ax.legend()
fig.tight_layout()
plt.show()

**Nhận xét:** Trong các sản phẩm có bán năm 2019, top 10% chiếm 64,04% dòng mua và top 20% chiếm 81,99%.

**Quy luật đề xuất:** Chọn sản phẩm theo trọng số tập trung vào nhóm bán chạy, dùng nhất quán cho đơn hàng và log; chưa đủ căn cứ chọn dạng phân phối tham số.

### 2.3. Nhóm hàng nào được mua nhiều?

Tỷ trọng trên toàn bộ dòng mua năm 2019; biểu đồ hiển thị 8 nhóm lớn nhất.

In [ ]:
hm_articles = pl.scan_csv(
    data_dir / "hm" / "articles.csv", infer_schema=False,
).select("article_id", "product_group_name").collect(engine="streaming")

group_demand = product_counts.join(
    hm_articles, on="article_id", how="left", validate="m:1",
).group_by("product_group_name").agg(
    pl.col("purchases").sum(),
).sort("purchases", descending=True).with_columns(
    share_pct=100.0 * pl.col("purchases") / pl.col("purchases").sum(),
)

group_demand.select("product_group_name", "purchases", pl.col("share_pct").round(2))

In [ ]:
top_groups = group_demand.head(8)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    x=top_groups["share_pct"].to_numpy(), y=top_groups["product_group_name"].to_list(),
    color="#4C72B0", ax=ax,
)
ax.set(xlabel="% dòng mua", ylabel="", title="Nhóm sản phẩm — H&M 2019")
fig.tight_layout()
plt.show()

**Nhận xét:** Nhóm thân trên chiếm 39,67% dòng mua, thân dưới 22,20%; nhu cầu khác nhau giữa các nhóm hàng.

**Quy luật đề xuất:** Dùng trọng số mua theo nhóm sau khi ánh xạ ngành hàng; tách cơ cấu nhu cầu khỏi cơ cấu số sản phẩm trong catalog.

### 2.4. Nhu cầu thay đổi theo tháng thế nào?

Dùng số dòng mua/ngày để loại ảnh hưởng tháng dài/ngắn. Chỉ số 1 là bình quân ngày của năm 2019.

In [ ]:
monthly_demand = hm_2019.group_by(
    pl.col("t_dat").dt.truncate("1mo").alias("month"),
).len(name="purchases").collect(engine="streaming").sort("month")

monthly_demand = monthly_demand.with_columns(
    days=pl.col("month").dt.month_end().dt.day(),
).with_columns(
    purchases_per_day=pl.col("purchases") / pl.col("days"),
).with_columns(
    demand_index=pl.col("purchases_per_day") / (pl.col("purchases").sum() / pl.col("days").sum()),
)

monthly_demand.select(
    "month", "purchases", pl.col("purchases_per_day").round(0), pl.col("demand_index").round(2),
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(monthly_demand["month"].dt.month(), monthly_demand["demand_index"], marker="o")
ax.axhline(1, color="gray", linestyle="--")
ax.set(xlabel="Tháng", ylabel="Chỉ số dòng mua/ngày", title="Nhu cầu theo tháng — H&M 2019")
ax.set_xticks(range(1, 13))
fig.tight_layout()
plt.show()

**Nhận xét:** Sau khi chuẩn hóa số ngày, chỉ số nhu cầu đạt khoảng 1,41 ở tháng 6 và 0,80 ở tháng 12, so với bình quân năm bằng 1.

**Quy luật đề xuất:** Cho phép nhu cầu nền thay đổi theo tháng; mùa vụ H&M 2019 chưa đại diện Việt Nam hoặc tách được tác động khuyến mãi/Tết.

## 3. eCommerce Behavior — sự kiện

In [ ]:
behavior_logs = pl.scan_csv(
    [
        data_dir / "ecommerce_behavior" / "2019-Oct.csv",
        data_dir / "ecommerce_behavior" / "2019-Nov.csv",
    ],
    infer_schema=False,
).select(
    "event_time", "event_type", "user_id", "user_session", "product_id",
    "price", "brand", "category_code",
).with_columns(
    pl.col("event_time").str.strip_suffix(" UTC").str.to_datetime(
        "%Y-%m-%d %H:%M:%S", time_zone="UTC", strict=False,
    ),
    pl.col("price").cast(pl.Float64, strict=False),
)

In [ ]:
behavior_logs.head()

In [ ]:
behavior_logs.select("price").collect(engine="streaming").describe()

In [ ]:
behavior_logs.null_count().collect(engine="streaming")

### 3.1. Phiên truy cập

Khóa phiên là `user_id` + `user_session`. Quan sát 1/10–1/11/2019 UTC; bỏ phiên chạm ngày đầu/cuối cửa sổ để hạn chế chuỗi bị cắt ở biên.

`duration_minutes` là khoảng cách giữa sự kiện đầu/cuối quan sát, không phải thời gian thực ở lại trang.

In [ ]:
session_key = ["user_id", "user_session"]
path_key = session_key + ["product_id"]

observed_events = behavior_logs.select(
    *path_key, "event_time", "event_type",
).filter(
    pl.col("user_id").is_not_null()
    & pl.col("user_session").is_not_null()
    & pl.col("product_id").is_not_null()
    & pl.col("event_time").is_between(
        pl.datetime(2019, 10, 1, time_zone="UTC"),
        pl.datetime(2019, 11, 2, time_zone="UTC"), closed="left",
    ),
)

session_stats = observed_events.group_by(session_key).agg(
    start=pl.col("event_time").min(),
    end=pl.col("event_time").max(),
    events=pl.len(),
    viewed_products=pl.col("product_id").filter(pl.col("event_type") == "view").n_unique(),
    has_view=(pl.col("event_type") == "view").any(),
    has_cart=(pl.col("event_type") == "cart").any(),
    has_purchase=(pl.col("event_type") == "purchase").any(),
).filter(
    (pl.col("start") >= pl.datetime(2019, 10, 2, time_zone="UTC"))
    & (pl.col("end") < pl.datetime(2019, 11, 1, time_zone="UTC")),
).with_columns(
    duration_minutes=(pl.col("end") - pl.col("start")).dt.total_seconds() / 60,
).collect(engine="streaming")

session_events = observed_events.join(
    session_stats.lazy().select(session_key), on=session_key, how="semi",
)

session_stats.select("events", "viewed_products", "duration_minutes").describe()

In [ ]:
session_stats.select(
    sessions=pl.len(),
    with_cart_pct=(100.0 * pl.col("has_cart").mean()).round(2),
    with_purchase_pct=(100.0 * pl.col("has_purchase").mean()).round(2),
)

In [ ]:
event_mix = session_events.group_by("event_type").len(name="events").collect(
    engine="streaming",
).sort("events", descending=True)

event_mix

**Nhận xét:** Cohort có 8.974.691 phiên; trung vị hai sự kiện và một sản phẩm được xem mỗi phiên. Phiên có cart chiếm 6,27%, có purchase chiếm 6,83%; các tỷ lệ này chưa yêu cầu thứ tự.

**Quy luật đề xuất:** Sinh hành trình gắn cùng người dùng/phiên và sản phẩm; khoảng cách sự kiện đầu–cuối chỉ là thời gian quan sát.

### 3.2. Funnel xem → thêm giỏ → mua

Cùng sản phẩm trong cùng phiên, timestamp tăng nghiêm ngặt. Mẫu số là phiên có `view`; khác với tỷ lệ phiên chỉ cần có cart/purchase ở trên.

Mỗi đường đi lấy view đầu tiên, cart đầu tiên sau view và purchase đầu tiên sau cart.

In [ ]:
cart_events = session_events.filter(pl.col("event_type") == "cart").select(
    *path_key, "event_time",
).collect(engine="streaming")
cart_products = cart_events.select(path_key).unique()

view_times = session_events.filter(pl.col("event_type") == "view").join(
    cart_products.lazy(), on=path_key, how="semi",
).group_by(path_key).agg(
    view_at=pl.col("event_time").min(),
).collect(engine="streaming")

cart_candidates = cart_events.join(view_times, on=path_key, how="inner")
cart_paths = cart_candidates.filter(pl.col("event_time") > pl.col("view_at")).group_by(
    path_key,
).agg(
    view_at=pl.col("view_at").first(),
    cart_at=pl.col("event_time").min(),
)

In [ ]:
purchase_events = session_events.filter(pl.col("event_type") == "purchase").select(
    *path_key, "event_time",
).collect(engine="streaming")

purchase_candidates = purchase_events.join(cart_paths, on=path_key, how="inner")
purchase_paths = purchase_candidates.filter(
    pl.col("event_time") > pl.col("cart_at"),
).group_by(path_key).agg(
    view_at=pl.col("view_at").first(),
    cart_at=pl.col("cart_at").first(),
    purchase_at=pl.col("event_time").min(),
)

funnel = pl.DataFrame({
    "stage": ["view", "view → cart", "view → cart → purchase"],
    "sessions": [
        session_stats["has_view"].sum(),
        cart_paths.select(session_key).unique().height,
        purchase_paths.select(session_key).unique().height,
    ],
}).with_columns(
    conversion_pct=100.0 * pl.col("sessions") / pl.col("sessions").first(),
)

funnel.with_columns(pl.col("conversion_pct").round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
sns.barplot(
    x=funnel["conversion_pct"].to_numpy(), y=funnel["stage"].to_list(),
    color="#4C72B0", ax=ax,
)
ax.bar_label(ax.containers[0], fmt="%.2f%%", padding=3)
ax.set(xlabel="% phiên có view", ylabel="", title="Funnel có thứ tự — 10/2019", xlim=(0, 110))
fig.tight_layout()
plt.show()

Các đường đi có cart trùng giờ view đầu tiên hoặc purchase trùng giờ cart đầu tiên được đếm riêng dưới đây; timestamp đó không chứng minh được thứ tự.

In [ ]:
pl.DataFrame({
    "same_timestamp": ["view/cart", "cart/purchase"],
    "session_product_paths": [
        cart_candidates.filter(pl.col("event_time") == pl.col("view_at")).select(path_key).unique().height,
        purchase_candidates.filter(pl.col("event_time") == pl.col("cart_at")).select(path_key).unique().height,
    ],
})

**Nhận xét:** Trong các phiên có view, 6,23% đạt view → cart và 3,10% đạt đủ view → cart → purchase cùng sản phẩm, đúng thứ tự. Có 2.623 đường đi trùng timestamp view/cart; không suy thứ tự từ các timestamp bằng nhau.

**Quy luật đề xuất:** Sinh chuỗi hành động có thứ tự và đo bằng cùng định nghĩa; không dùng trực tiếp tỷ lệ phiên làm trọng số request hoặc KPI mart hiện tại.

### 3.3. Thời gian giữa các bước

Tính trên đường đi `(khách, phiên, sản phẩm)` hoàn thành đủ ba bước. Đây là thời gian của nhóm có mua, không đại diện nhóm bỏ giỏ.

In [ ]:
conversion_times = purchase_paths.select(
    view_to_cart_minutes=(pl.col("cart_at") - pl.col("view_at")).dt.total_seconds() / 60,
    cart_to_purchase_minutes=(pl.col("purchase_at") - pl.col("cart_at")).dt.total_seconds() / 60,
    view_to_purchase_minutes=(pl.col("purchase_at") - pl.col("view_at")).dt.total_seconds() / 60,
)

conversion_times.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=conversion_times["view_to_purchase_minutes"].to_numpy(), ax=ax)
ax.set_xscale("log")
ax.set(xlabel="Phút từ xem đến mua (thang log)", ylabel="Tỷ lệ đường đi tích lũy", title="Thời gian chuyển đổi — 10/2019")
fig.tight_layout()
plt.show()

**Nhận xét:** Trong 296.130 đường đi hoàn thành, trung vị xem → giỏ là 0,45 phút, giỏ → mua 1,35 phút và xem → mua 2,18 phút.

**Quy luật đề xuất:** Sinh khoảng chờ giữa hành động cho hành trình mua hoàn thành; không dùng các khoảng chờ này làm HTTP latency hoặc đại diện nhóm bỏ giỏ.

## 4. Quy luật tổng hợp

| Nhóm | Hướng sinh dữ liệu |
|---|---|
| Nhu cầu | Tần suất mua, độ phổ biến sản phẩm, nhóm hàng và nhu cầu theo tháng có trọng số khác nhau. |
| Đơn hàng | Giỏ nhỏ chiếm đa số; giữ phần đơn giá trị cao và phân biệt số sản phẩm với số lượng. |
| Vận hành | Mô hình hóa ngày hẹn/giao thực tế và tham khảo đánh giá theo trải nghiệm giao nhận. |
| Hành vi | Giữ người dùng/phiên/sản phẩm xuyên chuỗi hành động, có thứ tự và khoảng chờ. |

**Thiếu bằng chứng:** Giá VND, địa lý, COD/VietQR, Tết/chiến dịch VN; tỷ lệ tồn kho, đổi trả, coupon/POS, lỗi HTTP và latency chưa được đo từ ba nguồn này.

**Phạm vi:** Đây là quy luật tham chiếu để đề xuất hiệu chỉnh; chưa thay generator hoặc chứng minh dữ liệu sinh đã khớp. Cần so sánh đầu ra trước/sau và kiểm tra trên giai đoạn dữ liệu để riêng.

**Phạm vi hiện tại:** Generator chỉ sinh SQL; EDA hành vi giữ để tham khảo, chưa triển khai sinh log.